# 🌳 Hierarchical Clustering — Demo Notebook

This notebook demonstrates agglomerative hierarchical clustering using
**scikit-learn** (`AgglomerativeClustering`, `FeatureAgglomeration`) and
**SciPy** (`linkage`, `dendrogram`, `fcluster`) — both core dependencies,
no special environment required.

**Companion reference:** `hierarchical_clustering_cheatsheet.md`


## 📦 Imports & Environment Check

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
import scipy

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, FeatureAgglomeration
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import kneighbors_graph

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

print(f"scikit-learn version: {sklearn.__version__}")
print(f"scipy version:         {scipy.__version__}")


## 🍷 Dataset

Using the built-in **wine** dataset (13 continuous chemical-analysis
features, 3 known cultivar classes) — consistent with the other wine-based
practice notebooks in this library.


In [ ]:
wine = load_wine()
X = wine.data
y_true = wine.target  # ground-truth cultivar, used only for evaluation
feature_names = wine.feature_names

df = pd.DataFrame(X, columns=feature_names)
df["true_cultivar"] = y_true

print(df.shape)
df.head()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2D projection for visualization only -- clustering itself uses all 13 features
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance (2 components): {pca.explained_variance_ratio_.sum():.2%}")


## 🌿 Build the Dendrogram (SciPy)

Building the full merge tree first, before committing to any number of
clusters, is the main advantage hierarchical clustering has over
KMeans/KMedoids.


In [ ]:
Z = linkage(X_scaled, method="ward")
print(f"Linkage matrix shape: {Z.shape}  (n_samples - 1, 4)")
print("First few merges (cluster_a, cluster_b, distance, size):")
print(Z[:5])


In [ ]:
plt.figure(figsize=(11, 5))
dendrogram(Z, truncate_mode="lastp", p=20, leaf_rotation=90, leaf_font_size=9)
plt.axhline(y=18, color="red", linestyle="--", label="Example cut at distance=18")
plt.title("Ward Linkage Dendrogram (last 20 merges)")
plt.xlabel("Cluster size or sample index")
plt.ylabel("Merge distance")
plt.legend()
plt.tight_layout()
plt.show()


> 💡 Look for the tallest vertical gap the horizontal cut line can pass
> through without crossing a merge line — that gap suggests a natural
> number of clusters. Here, cutting near distance ≈ 18 separates the tree
> into 3 groups, matching the known number of cultivars.


## 🔢 Cut the Tree Two Ways: by k and by distance

In [ ]:
# Cut for exactly 3 clusters
labels_k3 = fcluster(Z, t=3, criterion="maxclust")

# Cut at a specific merge distance
labels_dist = fcluster(Z, t=18.0, criterion="distance")

print(f"Clusters from maxclust=3:        {len(np.unique(labels_k3))} clusters")
print(f"Clusters from distance cut=18.0: {len(np.unique(labels_dist))} clusters")
print(f"ARI vs true cultivars (k=3 cut): {adjusted_rand_score(y_true, labels_k3):.3f}")


## 🔵 scikit-learn `AgglomerativeClustering` (Flat Labels)

Confirms scikit-learn's flat-label output agrees with the SciPy tree cut
above.


In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)

print(f"Silhouette score: {silhouette_score(X_scaled, agg_labels):.3f}")
print(f"ARI vs true cultivars: {adjusted_rand_score(y_true, agg_labels):.3f}")
print(f"ARI vs SciPy fcluster(k=3) labels: {adjusted_rand_score(labels_k3, agg_labels):.3f}")


> The near-1.0 ARI between the scikit-learn labels and the SciPy
> `fcluster` labels confirms both are doing the same underlying
> computation — pick whichever API fits the surrounding code (SciPy for
> visualization-first exploration, scikit-learn for pipeline integration).


## 📊 Visualize Cluster Assignments (PCA projection)

In [ ]:
plt.figure(figsize=(6.5, 5.5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=agg_labels, cmap="viridis", s=40, alpha=0.85)
plt.title("Agglomerative Clustering (ward linkage, k=3)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Cluster")
plt.tight_layout()
plt.show()


## 🔗 Linkage Method Comparison

Same data, same k, four different linkage rules — showing how much the
merge criterion alone changes the result.


In [ ]:
linkage_methods = ["ward", "complete", "average", "single"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

results = {}
for ax, method in zip(axes, linkage_methods):
    # ward requires euclidean; the others default to euclidean here too for a fair comparison
    model = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels = model.fit_predict(X_scaled)
    results[method] = labels

    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="viridis", s=25, alpha=0.85)
    ari = adjusted_rand_score(y_true, labels)
    sizes = np.bincount(labels)
    ax.set_title(f"{method}\nARI={ari:.3f}, sizes={list(sizes)}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()


> 💡 Watch the cluster **sizes** printed in each subplot title, not just
> the ARI. `single` linkage in particular tends to produce one dominant
> cluster plus tiny leftover clusters — the classic signature of the
> chaining effect described in the cheatsheet.


## 🔢 Choosing k Quantitatively (Silhouette Score)

In [ ]:
scores = {}
for k in range(2, 10):
    labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

best_k = max(scores, key=scores.get)

plt.figure(figsize=(7, 4.5))
plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"Best k = {best_k}")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score vs k (ward linkage)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best k by silhouette score: {best_k} (score={scores[best_k]:.3f})")


## 🔗 Connectivity Constraint Demo

Restricting merges to a k-nearest-neighbors graph — useful when you want
to enforce local structure (e.g., only nearby points may merge directly).


In [ ]:
connectivity = kneighbors_graph(X_scaled, n_neighbors=10, include_self=False)

agg_connected = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward",
    connectivity=connectivity,
)
labels_connected = agg_connected.fit_predict(X_scaled)

print(f"Connected components in the constrained graph: {agg_connected.n_connected_components_}")
print(f"ARI vs unconstrained ward clustering: {adjusted_rand_score(agg_labels, labels_connected):.3f}")


> ⚠️ If `n_connected_components_` exceeds `n_clusters`, the connectivity
> graph split the data into more pieces than requested — either loosen
> the graph (increase `n_neighbors`) or accept the extra fragmentation.


## 🧩 Feature Agglomeration

Clustering the *columns* instead of the rows — a lightweight,
interpretable alternative to PCA for dimensionality reduction.


In [ ]:
fa = FeatureAgglomeration(n_clusters=5, linkage="ward")
X_reduced = fa.fit_transform(X_scaled)

print(f"Original feature count: {X_scaled.shape[1]}")
print(f"Reduced feature count:  {X_reduced.shape[1]}")

# Which original features got grouped into each reduced feature?
for cluster_id in range(fa.n_clusters_):
    grouped = [f for f, c in zip(feature_names, fa.labels_) if c == cluster_id]
    print(f"  Reduced feature {cluster_id}: {grouped}")


## ⏱️ Scalability Check

A quick look at how fit time grows with `n` — illustrating the O(n²)
memory/time caution from the cheatsheet. Uses resampled copies of the
wine dataset to simulate larger n.


In [ ]:
import time

rng = np.random.default_rng(42)
sizes = [178, 356, 712, 1424]  # 1x, 2x, 4x, 8x the base wine dataset, via resampling with replacement
timings = []

for n in sizes:
    idx = rng.integers(0, X_scaled.shape[0], size=n)
    X_sim = X_scaled[idx]

    t0 = time.perf_counter()
    AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X_sim)
    elapsed = time.perf_counter() - t0
    timings.append(elapsed)
    print(f"n={n:5d}  fit time={elapsed*1000:7.1f} ms")

plt.figure(figsize=(6.5, 4.5))
plt.plot(sizes, timings, marker="o")
plt.xlabel("n_samples")
plt.ylabel("Fit time (seconds)")
plt.title("Agglomerative Clustering: Fit Time vs n_samples")
plt.tight_layout()
plt.show()


> The upward curve here is the practical face of the O(n²) memory / O(n³)
> naive time complexity noted in the cheatsheet — this is why hierarchical
> clustering is a small-to-medium-data tool without extra tricks
> (subsampling, connectivity constraints, or approximate methods).


## 📋 Results Summary

| Method | ARI vs true cultivars | Notes |
|---|---|---|
| SciPy `fcluster` (k=3 cut) | see output above | Matches scikit-learn's flat labels |
| scikit-learn `AgglomerativeClustering` (ward) | see output above | Best-performing linkage on this dataset |
| `complete` linkage | see output above | Tighter, more balanced clusters than `single` |
| `average` linkage | see output above | Middle ground |
| `single` linkage | see output above | Watch for chaining — check cluster sizes above |

**Takeaway:** on this clean, continuous, well-scaled dataset, `ward`
linkage recovers the three cultivars about as well as KMeans did in the
companion KMeans-vs-KMedoids notebook — but hierarchical clustering
additionally hands you the full dendrogram, letting you inspect the
merge structure and choose a cut point without committing to k in
advance. The trade-off is O(n²) memory, which caps practical dataset
size compared to KMeans/KMedoids.

See `hierarchical_clustering_cheatsheet.md` for the full reference,
including linkage method details, connectivity constraints, and feature
agglomeration.
